# Examples for Name Matching (using Polars)

This notebook illustrates basic usage of the entity matching model with Polars DataFrames. Install the optional dependency with `pip install "emm[polars]"`.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import joblib
import polars as pl
from emm import PolarsEntityMatching

In [ ]:
gt = pl.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'name': ['John Smith LLC', 'ING LLC', 'John Doe LLC', 'Zhe Sun G.M.B.H', 'Random GMBH'],
})
gt

Prepare a small supervised model for illustration. For production, train on representative labeled data.

In [ ]:
workspace = TemporaryDirectory()
model_dir = Path(workspace.name)
train_names = pl.DataFrame({
    'id': [1, 2, 3, 4, 5, 1, 3],
    'name': ['John Smith', 'I.n.G. LLC', 'Jon DOEE LLC', 'Zhe Sun', 'Random', 'J Smith', 'John Doe'],
})
training_model = PolarsEntityMatching(
    name_only=True,
    indexers=[{'type': 'cosine_similarity', 'tokenizer': 'characters', 'ngram': 2,
               'num_candidates': 5, 'cos_sim_lower_bound': 0.2}],
    n_jobs=1,
)
training_model.fit(gt)
training_model.fit_classifier(train_names, create_negative_sample_fraction=0.5, n_jobs=1)
supervised_model = training_model.supervised_models['nm_score']['model']
joblib.dump(supervised_model, model_dir / 'sem_nm.pkl')

## Name matching without a supervised model

Use word tokenization and cosine similarity. The candidate score is the cosine similarity.

In [ ]:
queries = pl.DataFrame({
    'id': [10, 11, 12],
    'name': ['John Smith', 'I.n.G. LLC', 'Jon DOEE LLC'],
})
nm = PolarsEntityMatching({
    'name_only': True,
    'entity_id_col': 'id',
    'name_col': 'name',
    'preprocessor': 'preprocess_merge_abbr',
    'indexers': [{'type': 'cosine_similarity', 'tokenizer': 'words', 'ngram': 1,
                  'num_candidates': 5, 'cos_sim_lower_bound': 0.2}],
    'supervised_on': False,
    'n_jobs': 1,
})
nm.fit(gt)
res = nm.transform(queries)
assert res.filter((pl.col('name') == 'John Smith') & (pl.col('gt_entity_id') == 1)).height > 0
res

Use 2-character n-grams and cosine similarity without a supervised model.

In [ ]:
nm = PolarsEntityMatching({
    'name_only': True,
    'entity_id_col': 'id',
    'name_col': 'name',
    'preprocessor': 'preprocess_merge_abbr',
    'indexers': [{'type': 'cosine_similarity', 'tokenizer': 'characters', 'ngram': 2,
                  'num_candidates': 5, 'cos_sim_lower_bound': 0.2}],
    'supervised_on': False,
    'n_jobs': 1,
})
nm.fit(gt)
res = nm.transform(queries)
res

Combine word and character cosine-similarity indexers without a supervised model.

In [ ]:
nm = PolarsEntityMatching({
    'name_only': True,
    'entity_id_col': 'id',
    'name_col': 'name',
    'preprocessor': 'preprocess_merge_abbr',
    'indexers': [
        {'type': 'cosine_similarity', 'tokenizer': 'words', 'ngram': 1,
         'num_candidates': 5, 'cos_sim_lower_bound': 0.2},
        {'type': 'cosine_similarity', 'tokenizer': 'characters', 'ngram': 2,
         'num_candidates': 5, 'cos_sim_lower_bound': 0.2},
    ],
    'supervised_on': False,
    'n_jobs': 1,
})
nm.fit(gt)
res = nm.transform(queries)
res

Use sorted-neighbourhood indexing (SNI). The score is a normalized SNI distance.

In [ ]:
nm = PolarsEntityMatching({
    'name_only': True,
    'uid_col': 'uid',
    'entity_id_col': 'id',
    'name_col': 'name',
    'preprocessor': 'preprocess_merge_abbr',
    'indexers': [{'type': 'sni', 'window_length': 3}],
    'supervised_on': False,
})
nm.fit(gt)
res = nm.transform(pl.DataFrame({'id': [10, 11, 12], 'name': ['Jo S', 'InG. LLC', 'Jon DOEE LLC']}))
res

A custom function can transform names before SNI, for example by reversing them.

In [ ]:
def reverse_name(name):
    return name[::-1]

nm = PolarsEntityMatching({
    'name_only': True,
    'uid_col': 'uid',
    'entity_id_col': 'id',
    'name_col': 'name',
    'preprocessor': 'preprocess_merge_abbr',
    'indexers': [{'type': 'sni', 'window_length': 3, 'mapping_func': reverse_name}],
    'supervised_on': False,
})
nm.fit(gt)
res = nm.transform(pl.DataFrame({
    'id': [11, 12, 13], 'name': ['a InG. LLC', 'ING. LLC ZZZ', 'John Smith LLC'],
}))
res

A blocking function restricts candidate pairs to names with the same blocking value.

In [ ]:
def first_character(name):
    return name[0] if name else '?'

nm = PolarsEntityMatching({
    'name_only': True,
    'uid_col': 'uid',
    'entity_id_col': 'id',
    'name_col': 'name',
    'preprocessor': 'preprocess_merge_abbr',
    'indexers': [{'type': 'cosine_similarity', 'tokenizer': 'characters', 'ngram': 1,
                  'blocking_func': first_character}],
    'supervised_on': False,
    'n_jobs': 1,
})
nm.fit(gt)
res = nm.transform(pl.DataFrame({'id': [10, 11], 'name': ['!notING', 'ING']}))
res

## Name matching with a supervised model

Load the illustrative classifier trained above and score the candidate pairs.

In [ ]:
nm = PolarsEntityMatching({
    'name_only': True,
    'entity_id_col': 'id',
    'name_col': 'name',
    'preprocessor': 'preprocess_merge_abbr',
    'indexers': [{'type': 'cosine_similarity', 'tokenizer': 'characters', 'ngram': 2,
                  'num_candidates': 5, 'cos_sim_lower_bound': 0.2}],
    'supervised_on': True,
    'supervised_model_filename': 'sem_nm.pkl',
    'supervised_model_dir': model_dir,
    'n_jobs': 1,
})
nm.fit(gt)
res = nm.transform(queries)
assert 'nm_score' in res.columns and res.height > 0
res

## Name matching using multiple indexers

Combine word cosine similarity, character cosine similarity and SNI.

In [ ]:
nm = PolarsEntityMatching({
    'name_only': True,
    'entity_id_col': 'id',
    'name_col': 'name',
    'preprocessor': 'preprocess_merge_abbr',
    'indexers': [
        {'type': 'cosine_similarity', 'tokenizer': 'words', 'ngram': 1,
         'num_candidates': 5, 'cos_sim_lower_bound': 0.2},
        {'type': 'cosine_similarity', 'tokenizer': 'characters', 'ngram': 2,
         'num_candidates': 5, 'cos_sim_lower_bound': 0.2},
        {'type': 'sni', 'window_length': 3},
    ],
    'supervised_on': False,
    'n_jobs': 1,
})
nm.fit(gt)
res = nm.transform(pl.DataFrame({
    'id': [10, 11, 12, 14], 'name': ['John Smith', 'I.n.G. LLC', 'Jon DOEE LLC', 'Z'],
}))
res

## Name matching from a serialized model

A loaded model can match names without refitting the ground-truth indexers. This example stores files in a temporary directory.

In [ ]:
model_path = model_dir / 'name_matching_polars_model.joblib'
nm.save(str(model_path))
nm2 = PolarsEntityMatching.load(str(model_path))
saved_queries = pl.DataFrame({'id': [10, 11, 12], 'name': ['John Smith', 'I.n.G. LLC', 'Jon DOE LLC']})
original_results = nm.transform(saved_queries)
res2 = nm2.transform(saved_queries)
assert res2.equals(original_results)
res2

In [ ]:
workspace.cleanup()